# Reasoning Memory: Colab
Включите GPU runtime. Первая ячейка клонирует или обновляет `/content/reasoning-memory` через fast-forward.
Для приватного репозитория добавьте в Colab Secrets `GITHUB_TOKEN` с правом **Contents: Read-only** и разрешите доступ ноутбуку.
Без секрета появится скрытый ввод токена. Повторный запуск сохраняет папки `runs/`.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile
from getpass import getpass
from google.colab import files, userdata

REPO_URL = 'https://github.com/Ferraronp/reasoning-memory.git'
BRANCH = 'main'
PROJECT_ROOT = Path('/content/reasoning-memory')

def sync_project(repo_url, branch, project_root, git_env=None):
    project_root = Path(project_root)
    def git(*args, cwd=None):
        result = subprocess.run(['git', '-c', 'credential.helper=', *args], cwd=cwd,
                                env=git_env, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError(result.stderr.strip() or result.stdout.strip() or 'Git failed')
        return result.stdout.strip()
    if not project_root.exists():
        git('clone', '--branch', branch, '--single-branch', repo_url, str(project_root))
    else:
        if not (project_root / '.git').is_dir():
            raise RuntimeError(f'{project_root} exists but is not a git clone. Choose another PROJECT_ROOT.')
        origin = git('remote', 'get-url', 'origin', cwd=project_root)
        if origin.rstrip('/') != repo_url.rstrip('/'):
            raise RuntimeError('The existing folder has a different origin; choose another PROJECT_ROOT.')
        if git('branch', '--show-current', cwd=project_root) != branch:
            raise RuntimeError(f'The checkout is not on {branch}. Switch it manually before updating.')
        if git('status', '--porcelain', '--untracked-files=no', cwd=project_root):
            raise RuntimeError('Local source changes found. Commit/stash them before updating; nothing was overwritten.')
        git('fetch', 'origin', branch, cwd=project_root)
        git('merge', '--ff-only', 'FETCH_HEAD', cwd=project_root)
    print('Project:', project_root)
    print('Commit:', git('log', '-1', '--format=%h %s', cwd=project_root))

def sync_private_project():
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    if not token:
        token = getpass('GitHub token (read-only Contents for this repository): ').strip()
    if not token:
        raise ValueError('A token is required for this private repository')
    # Helper contains only environment-variable references, never the actual token.
    with tempfile.TemporaryDirectory(prefix='rm-git-auth-') as auth_dir:
        askpass = Path(auth_dir) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" "x-access-token" ;;\n  *) printf "%s\\n" "$RM_GITHUB_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0', RM_GITHUB_TOKEN=token)
        try:
            sync_project(REPO_URL, BRANCH, PROJECT_ROOT, env)
        finally:
            env.pop('RM_GITHUB_TOKEN', None)
            token = None

sync_private_project()
assert (PROJECT_ROOT / 'pyproject.toml').is_file()


## Установка и краткая проверка
`doctor` покажет версию, коммит и GPU. Полный список библиотек сохраняется в `runs/.../environment.json`.


In [ ]:
def run_process(argv):
    # Pipes are read in Python so Jupyter/Colab reliably displays child output.
    with subprocess.Popen(argv, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        return process.wait()

if run_process([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', f'{PROJECT_ROOT}[hf]']) != 0:
    raise RuntimeError('Installation failed; see output above')

def run_cli(*args):
    code = run_process([sys.executable, '-u', '-m', 'reasoning_memory', *map(str, args)])
    if code not in (0, 2):
        raise RuntimeError(f'Ошибка запуска, exit={code}; см. вывод выше')
    if code == 2:
        print('Запуск закончен с диагностическими ошибками. Причина и текст генерации показаны выше.')
    # Intentionally no CompletedProcess return value.

run_cli('doctor')


## Следующая проверка: несколько промежуточных результатов
Первый этап вычисляет несколько значений и сохраняет короткий вывод. Второй этап появляется только после разделения на `full` и `compact` и комбинирует эти значения.
Начните с `LIMIT = 1`: две линейные функции, затем `f(g(5))`, ожидаем `87`. При `LIMIT = 3` дополнительно проверяются зависимые величины (`222`) и кратчайший путь (`45`).
Просматривайте содержимое первого вывода. Если в нём ошибочное или неполное значение, финальная ошибка не доказывает вред от свёртки.
Исходная постановка остаётся доступной обеим веткам, поэтому правильный ответ может получиться и повторным вычислением. Один прогон при температуре 0.6 — диагностический пример, а не оценка качества метода.


In [ ]:
LIMIT = 1  # Затем 3 — все три задачи
run_cli('pair', '--config', 'configs/qwen3_17b_fp16_chat_two_stage.json',
        '--tasks', 'data/diagnostic_reuse.jsonl', '--limit', LIMIT)


## Что именно сохранилось и сколько контекста нужно
Показываем первый вывод, исход второго этапа, длину входа перед ним и итоговый контекст. Ответ проверяется строгим сравнением с ожидаемой строкой.


In [ ]:
latest = max((PROJECT_ROOT / 'runs').iterdir(), key=lambda p: p.name)
manifest = json.loads((latest / 'manifest.json').read_text())
tasks = json.loads((latest / 'tasks.json').read_text())
print('Run:', latest.name, '|', manifest['protocol'], '|', manifest['status'])
for index, task in enumerate(tasks):
    folder = latest / f'task_{index:05d}'
    prefix = json.loads((folder / 'shared_prefix.json').read_text())
    print('\nTASK', task['id'], '| expected:', task.get('expected'))
    print('Saved conclusion:', prefix.get('archive', {}).get('e1', {}).get('summary', '(missing)'))
    events = [json.loads(line) for line in (folder / 'events.jsonl').read_text().splitlines()]
    for mode in ('full', 'compact'):
        path = folder / f'{mode}.json'
        if not path.exists():
            print(mode, 'not created; inspect shared events')
            continue
        result = json.loads(path.read_text())
        stage2 = next((e for e in events if e['branch'] == mode and e.get('stage') == 2), None)
        stage2_tokens = stage2.get('generation', {}).get('input_tokens') if stage2 else None
        print(mode, '| status:', result['status'], '| answer:', repr(result['answer']),
              '| exact:', result['correct'], '| stage2 input:', stage2_tokens,
              '| final active:', result['active_tokens'], '| experiments:', result['experiments'])
print('\nSummary:', (latest / 'summary.json').read_text())


## Скачать результаты
Диск Colab временный. Архив включает все запуски из `runs/`.


In [ ]:
import shutil
result_zip = shutil.make_archive('/content/reasoning-memory-results', 'zip', PROJECT_ROOT, 'runs')
files.download(result_zip)
